展示所有dcm文件的类型，判断每个 DICOM 是灰度还是彩色

第二批结果：全部图像，每个像素都有 3 个颜色通道，属于真正的 RGB 彩色图像，而不是灰度 DICOM，也不是 PALETTE COLOR 伪彩色 DICOM

In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd

# 如果没有安装 pydicom，先运行：
# pip install pydicom
import pydicom


# =========================
# 1. 设置根目录
# =========================
root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_1\raw")


# =========================
# 2. 判断 DICOM 类型
# =========================
def classify_dicom(dcm_path):
    try:
        ds = pydicom.dcmread(str(dcm_path), stop_before_pixels=True, force=True)

        photometric = str(getattr(ds, "PhotometricInterpretation", "Unknown"))
        samples_per_pixel = getattr(ds, "SamplesPerPixel", "Unknown")
        rows = getattr(ds, "Rows", "Unknown")
        cols = getattr(ds, "Columns", "Unknown")

        # 常见彩色 DICOM
        if samples_per_pixel == 3 or photometric in [
            "RGB",
            "YBR_FULL",
            "YBR_FULL_422",
            "YBR_PARTIAL_422",
            "PALETTE COLOR"
        ]:
            dicom_type = "彩色DICOM"

        # 常见灰度 DICOM
        elif samples_per_pixel == 1 or photometric in [
            "MONOCHROME1",
            "MONOCHROME2"
        ]:
            dicom_type = "灰度DICOM"

        else:
            dicom_type = "无法确定"

        return {
            "path": str(dcm_path),
            "dicom_type": dicom_type,
            "PhotometricInterpretation": photometric,
            "SamplesPerPixel": samples_per_pixel,
            "Rows": rows,
            "Columns": cols,
            "error": ""
        }

    except Exception as e:
        return {
            "path": str(dcm_path),
            "dicom_type": "读取失败",
            "PhotometricInterpretation": "",
            "SamplesPerPixel": "",
            "Rows": "",
            "Columns": "",
            "error": str(e)
        }


# =========================
# 3. 递归扫描所有 dcm 文件
# =========================
dcm_files = list(root_dir.rglob("*.dcm"))

print(f"共找到 {len(dcm_files)} 个 DICOM 文件。\n")

results = []

for dcm_path in dcm_files:
    results.append(classify_dicom(dcm_path))

df = pd.DataFrame(results)


# =========================
# 4. 统计结果
# =========================
print("DICOM 类型统计：")
print(df["dicom_type"].value_counts())
print()


print("PhotometricInterpretation 统计：")
print(df["PhotometricInterpretation"].value_counts())
print()


print("SamplesPerPixel 统计：")
print(df["SamplesPerPixel"].value_counts())
print()


# =========================
# 5. 展示详细结果
# =========================
df

共找到 2801 个 DICOM 文件。

DICOM 类型统计：
dicom_type
彩色DICOM    2801
Name: count, dtype: int64

PhotometricInterpretation 统计：
PhotometricInterpretation
RGB    2801
Name: count, dtype: int64

SamplesPerPixel 统计：
SamplesPerPixel
3    2801
Name: count, dtype: int64



,path,dicom_type,PhotometricInterpretation,SamplesPerPixel,Rows,Columns,error
0,G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\b...,彩色DICOM,RGB,3,1080,1920,
1,G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\b...,彩色DICOM,RGB,3,1080,1920,
2,G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\b...,彩色DICOM,RGB,3,1080,1920,
3,G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\b...,彩色DICOM,RGB,3,1080,1920,
4,G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\b...,彩色DICOM,RGB,3,1080,1920,
...,...,...,...,...,...,...,...
2796,G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\b...,彩色DICOM,RGB,3,768,1024,
2797,G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\b...,彩色DICOM,RGB,3,768,1024,
2798,G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\b...,彩色DICOM,RGB,3,768,1024,
2799,G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\b...,彩色DICOM,RGB,3,768,1024,


检查dcm位深

第二批图像  2801 个 DICOM 都是标准 8-bit RGB 彩色 DICOM
因此它们本身已经满足：
彩色三通道
像素范围通常为 0~255
适合直接保存为 RGB PNG
后续转换时不需要再做 min-max 或百分位归一化，否则反而可能改变颜色分布。



In [2]:
from pathlib import Path
from collections import Counter
import pydicom

root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_1\raw")

bits_counter = Counter()
dtype_counter = Counter()

for dcm_path in root_dir.rglob("*.dcm"):
    ds = pydicom.dcmread(str(dcm_path), force=True)
    bits_counter[getattr(ds, "BitsAllocated", "Unknown")] += 1
    dtype_counter[str(ds.pixel_array.dtype)] += 1

print("BitsAllocated 统计：")
print(bits_counter)

print("\npixel_array dtype 统计：")
print(dtype_counter)

BitsAllocated 统计：
Counter({8: 2801})

pixel_array dtype 统计：
Counter({'uint8': 2801})
